# Logistic Regression NIST Cross-Dataset Validation

This notebook mirrors the NIST validation workflow used for the Neural Network model, but applies it to the Logistic Regression baseline model.

Validation goal:
1. Train the logistic regression model using the NFI labeled particle data.
2. Reformat the NIST data so its element columns match the training feature space.
3. Run the model on NIST non-shooter particles as a false-positive / specificity test.
4. Run the model on NIST shooter particles as a sensitivity contrast.
5. Compare predicted GSR probabilities by source and class.

**Important:** This notebook keeps the same leakage-control logic as the logistic regression notebook by using a group-aware split on `stub_id`. It also keeps `pb`, `sb`, and `ba` excluded to match the current baseline modeling approach.


In [ ]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 150)


## 1. Load NFI Training Data

This is the same labeled particle dataset used in the logistic regression baseline notebook.


In [ ]:
# Adjust this path if needed based on where this notebook is saved
df = pd.read_parquet('../../data/processed/particle_labeled.parquet')

print(f'NFI labeled dataset shape: {df.shape}')
print('Label distribution:')
print(df['label'].value_counts(dropna=False))


## 2. Prepare Logistic Regression Features and Target

This cell matches the baseline logistic regression setup: binary labels only, excluded metadata/leakage fields, excludes `pb`, `sb`, and `ba`, and keeps numeric features only.


In [ ]:
# Copy the modeling dataframe
df_model = df.copy()

# Keep only binary labels
df_model = df_model[df_model['label'].isin(['GSR', 'Non_GSR'])].copy()

# Create binary target
df_model['target'] = df_model['label'].map({'Non_GSR': 0, 'GSR': 1})

# Columns to exclude from modeling
exclude_cols = [
    'stub_id', 'particle_id', 'label', 'target',
    'relevance_class', 'final_class', 'class', 'subclass', 'particle_class',
    'pb', 'sb', 'ba'
]

# Drop excluded columns if they exist
X = df_model.drop(columns=[col for col in exclude_cols if col in df_model.columns])

# Keep only numeric columns
X = X.select_dtypes(include=['number'])

y = df_model['target'].astype(int)
groups = df_model['stub_id']
training_features = X.columns.tolist()

print('Feature matrix shape:', X.shape)
print('Number of features:', X.shape[1])
print('Target distribution:')
print(y.value_counts(normalize=True))
print('
First 20 training features:')
print(training_features[:20])


## 3. Group-Aware Train/Test Split

The split prevents particles from the same `stub_id` from appearing in both the training and test sets.


In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]
groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print('Training shape:', X_train.shape)
print('Testing shape:', X_test.shape)
print('Unique training stubs:', groups_train.nunique())
print('Unique testing stubs:', groups_test.nunique())
print('Stub overlap:', len(set(groups_train).intersection(set(groups_test))))
print('
Training target distribution:')
print(y_train.value_counts(normalize=True))
print('
Testing target distribution:')
print(y_test.value_counts(normalize=True))


## 4. Train Baseline Logistic Regression Model

This uses the baseline settings from your current logistic regression notebook.


In [ ]:
baseline_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        solver='saga',
        C=0.5,
        class_weight='balanced',
        max_iter=5000,
        tol=0.001,
        random_state=42
    ))
])

baseline_pipe.fit(X_train, y_train)

baseline_pred = baseline_pipe.predict(X_test)
baseline_proba = baseline_pipe.predict_proba(X_test)[:, 1]

baseline_results = {
    'Model': 'Baseline Logistic Regression',
    'Regularization': 'L2',
    'C': 0.5,
    'Class Weight': 'balanced',
    'Accuracy': accuracy_score(y_test, baseline_pred),
    'Precision': precision_score(y_test, baseline_pred),
    'Recall': recall_score(y_test, baseline_pred),
    'F1': f1_score(y_test, baseline_pred),
    'ROC-AUC': roc_auc_score(y_test, baseline_proba),
    'PR-AUC': average_precision_score(y_test, baseline_proba)
}

pd.DataFrame([baseline_results])


## 5. Confirm Held-Out Test Performance


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(y_test, baseline_proba, ax=ax)
ax.set_title('Baseline Logistic Regression ROC Curve on NFI Test Set')
ax.grid(True, alpha=0.3)
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
PrecisionRecallDisplay.from_predictions(y_test, baseline_proba, ax=ax)
ax.set_title('Baseline Logistic Regression Precision-Recall Curve on NFI Test Set')
ax.grid(True, alpha=0.3)
plt.show()

print('Classification report at default threshold = 0.50')
print(classification_report(y_test, baseline_pred, target_names=['Non_GSR', 'GSR']))


## 6. Define Thresholds for NIST Validation

The default threshold is 0.50. Additional thresholds are selected from the NFI held-out test set.


In [ ]:
threshold_grid = np.linspace(0.01, 0.99, 99)
threshold_rows = []

for threshold in threshold_grid:
    threshold_pred = (baseline_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, threshold_pred).ravel()
    threshold_rows.append({
        'Threshold': threshold,
        'TN': tn,
        'FP': fp,
        'FN': fn,
        'TP': tp,
        'FPR': fp / (fp + tn) if (fp + tn) else np.nan,
        'FNR': fn / (fn + tp) if (fn + tp) else np.nan,
        'Precision': precision_score(y_test, threshold_pred, zero_division=0),
        'Recall': recall_score(y_test, threshold_pred, zero_division=0),
        'F1': f1_score(y_test, threshold_pred, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_rows)

def get_row_nearest_threshold(df, threshold):
    return df.loc[[(df['Threshold'] - threshold).abs().idxmin()]]

def get_best_f1_row(df):
    return df.loc[[df['F1'].idxmax()]]

def get_high_specificity_row(df, min_recall=0.70):
    candidates = df[df['Recall'] >= min_recall].copy()
    if candidates.empty:
        candidates = df.copy()
    idx = candidates.sort_values(['FPR', 'FN', 'Threshold'], ascending=[True, True, False]).index[0]
    return df.loc[[idx]]

def get_high_sensitivity_row(df, min_precision=0.10):
    candidates = df[df['Precision'] >= min_precision].copy()
    if candidates.empty:
        candidates = df.copy()
    idx = candidates.sort_values(['FNR', 'FP', 'Threshold'], ascending=[True, True, True]).index[0]
    return df.loc[[idx]]

default_row = get_row_nearest_threshold(threshold_df, 0.50).assign(Mode='Default Threshold')
best_f1_row = get_best_f1_row(threshold_df).assign(Mode='Balanced / Best F1')
high_specificity_row = get_high_specificity_row(threshold_df, min_recall=0.70).assign(Mode='High Specificity')
high_sensitivity_row = get_high_sensitivity_row(threshold_df, min_precision=0.10).assign(Mode='High Sensitivity')

threshold_summary = pd.concat([default_row, best_f1_row, high_specificity_row, high_sensitivity_row])
threshold_summary = threshold_summary[['Mode', 'Threshold', 'FPR', 'FNR', 'Precision', 'Recall', 'F1', 'FP', 'FN']].reset_index(drop=True)

thresholds = {row['Mode']: float(row['Threshold']) for _, row in threshold_summary.iterrows()}
threshold_summary


## 7. Load NIST Data

This is the same NIST parquet file used in the neural network validation notebook.


In [ ]:
# Adjust this path if needed based on where this notebook is saved
nist = pd.read_parquet('../../../data/processed/nist_concatenated_parquets/nist_neQuant_concatenated_data.parquet')

print(f'NIST shape: {nist.shape}')
print(f'Sample sources: {nist["sample_source"].nunique()}')
print('
Sample source counts:')
print(nist['sample_source'].value_counts().to_string())


## 8. Split NIST into Non-Shooter and Shooter Data


In [ ]:
shooter_mask = nist['sample_source'].str.startswith('shooter')
nist_nonshooter = nist[~shooter_mask].copy()
nist_shooter = nist[shooter_mask].copy()

print(f'Non-shooter particles: {len(nist_nonshooter):,} ({nist_nonshooter["sample_source"].nunique()} sources)')
print(f'Shooter particles: {len(nist_shooter):,} ({nist_shooter["sample_source"].nunique()} sources)')
print('
Non-shooter CLASS distribution:')
print(nist_nonshooter['CLASS'].value_counts().head(15).to_string())
print('
Shooter CLASS distribution:')
print(nist_shooter['CLASS'].value_counts().head(15).to_string())


## 9. Preprocess NIST into Logistic Regression Feature Format

NIST uses uppercase element columns and `sum_with_oxygen`. This converts NIST values into lowercase percentage columns, then aligns them to the exact logistic regression training features. Training features unavailable in NIST are filled with 0.


In [ ]:
nist_element_cols_upper = [
    'O','MG','AL','SI','P','S','CL','K','CA','TI','CR','MN',
    'FE','NI','CU','ZN','RB','SR','ZR','MO','RH','PD','AG',
    'IN','SN','SB','BA','LA','CE','ND','SM','TB','TM','AU','PB','BI'
]
nist_element_cols_upper = [col for col in nist_element_cols_upper if col in nist.columns]

def preprocess_nist_for_logistic(df, training_features):
    result = pd.DataFrame(index=df.index)
    if 'sum_with_oxygen' not in df.columns:
        raise ValueError('Expected NIST column sum_with_oxygen was not found.')

    sum_with_o = df['sum_with_oxygen'].replace(0, np.nan)
    for nist_col in nist_element_cols_upper:
        lower_name = nist_col.lower()
        result[lower_name] = (df[nist_col] / sum_with_o) * 100.0

    result['sample_source'] = df['sample_source'].values
    result['CLASS'] = df['CLASS'].values

    X_nist = result.reindex(columns=training_features, fill_value=0)
    X_nist = X_nist.apply(pd.to_numeric, errors='coerce').fillna(0)
    return result, X_nist

nist_nonshooter_proc, X_nist_nonshooter = preprocess_nist_for_logistic(nist_nonshooter, training_features)
nist_shooter_proc, X_nist_shooter = preprocess_nist_for_logistic(nist_shooter, training_features)

missing_training_features = sorted(set(training_features) - set(nist_nonshooter_proc.columns))
extra_nist_features = sorted(set(nist_nonshooter_proc.columns) - set(training_features) - {'sample_source', 'CLASS'})

print('NIST non-shooter model matrix:', X_nist_nonshooter.shape)
print('NIST shooter model matrix:', X_nist_shooter.shape)
print(f'Number of training features unavailable in NIST and filled with 0: {len(missing_training_features)}')
print(missing_training_features)
print(f'
NIST element columns not used by logistic model: {len(extra_nist_features)}')
print(extra_nist_features)


## 10. Run Logistic Regression on NIST Non-Shooter Data

Since these are non-shooter particles, predicted GSR classifications are treated as false positives.


In [ ]:
nonshooter_probs = baseline_pipe.predict_proba(X_nist_nonshooter)[:, 1]
print(f'Non-shooter predictions computed: {len(nonshooter_probs):,} particles')

for name, thresh in thresholds.items():
    preds = (nonshooter_probs >= thresh).astype(int)
    n_fp = preds.sum()
    fpr = n_fp / len(preds)
    print(f'Threshold={thresh:.4f} ({name}): {n_fp:,} false positives out of {len(preds):,} ({fpr:.4%})')


### Non-Shooter False Positives by Source


In [ ]:
sources = nist_nonshooter_proc['sample_source'].values
nonshooter_preds = (nonshooter_probs >= thresholds['Default Threshold']).astype(int)

source_results = []
for src in sorted(np.unique(sources)):
    mask = sources == src
    n = mask.sum()
    n_fp = nonshooter_preds[mask].sum()
    mean_p = nonshooter_probs[mask].mean()
    source_type = 'brakes' if 'brakes' in src else 'fireworks' if 'fireworks' in src else 'other'
    source_results.append({
        'Source': src,
        'Type': source_type,
        'N': n,
        'FP': n_fp,
        'FPR': n_fp / n,
        'Mean P(GSR)': mean_p
    })

source_results_df = pd.DataFrame(source_results)
display(source_results_df)

for stype in ['brakes', 'fireworks', 'other']:
    mask = source_results_df['Type'].eq(stype)
    if mask.any():
        n = source_results_df.loc[mask, 'N'].sum()
        n_fp = source_results_df.loc[mask, 'FP'].sum()
        print(f'{stype.upper()} total: {n_fp:,} FP out of {n:,} ({n_fp/n:.4%})')


### Non-Shooter False Positives by NIST Class


In [ ]:
if nonshooter_preds.sum() > 0:
    print('False-positive by NIST CLASS breakdown at default threshold = 0.50')
    fp_mask = nonshooter_preds == 1
    fp_classes = nist_nonshooter_proc['CLASS'].values[fp_mask]
    fp_class_counts = pd.Series(fp_classes).value_counts()
    display(fp_class_counts.rename('False Positives').to_frame())
else:
    print('No false positives in non-shooter data at default threshold = 0.50.')


## 11. Run Logistic Regression on NIST Shooter Data

This provides a sensitivity contrast. A model that generalizes well should assign higher GSR probabilities to shooter particles than to non-shooter particles.


In [ ]:
shooter_probs = baseline_pipe.predict_proba(X_nist_shooter)[:, 1]
print(f'Shooter predictions computed: {len(shooter_probs):,} particles')

for name, thresh in thresholds.items():
    preds = (shooter_probs >= thresh).astype(int)
    n_gsr = preds.sum()
    gsr_rate = n_gsr / len(preds)
    print(f'Threshold={thresh:.4f} ({name}): {n_gsr:,} classified as GSR out of {len(preds):,} ({gsr_rate:.2%})')


### Shooter GSR Rate by Source


In [ ]:
shooter_sources = nist_shooter_proc['sample_source'].values
shooter_preds = (shooter_probs >= thresholds['Default Threshold']).astype(int)

shooter_results = []
for src in sorted(np.unique(shooter_sources)):
    mask = shooter_sources == src
    n = mask.sum()
    n_gsr = shooter_preds[mask].sum()
    mean_p = shooter_probs[mask].mean()
    shooter_results.append({
        'Source': src,
        'N': n,
        'GSR': n_gsr,
        'GSR Rate': n_gsr / n,
        'Mean P(GSR)': mean_p
    })

shooter_results_df = pd.DataFrame(shooter_results)
display(shooter_results_df)


### Shooter GSR Rate by NIST Class


In [ ]:
shooter_classes = nist_shooter_proc['CLASS'].values
class_results = []

for cls in sorted(np.unique(shooter_classes)):
    mask = shooter_classes == cls
    n = mask.sum()
    if n < 10:
        continue
    n_gsr = shooter_preds[mask].sum()
    mean_p = shooter_probs[mask].mean()
    class_results.append({
        'CLASS': cls,
        'N': n,
        'GSR': n_gsr,
        'GSR Rate': n_gsr / n,
        'Mean P(GSR)': mean_p
    })

class_results_df = pd.DataFrame(class_results)
display(class_results_df)


## 12. Probability Distribution Comparison


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(nonshooter_probs, bins=100, alpha=0.7, label='Non-shooter', density=True)
axes[0].hist(shooter_probs, bins=100, alpha=0.7, label='Shooter', density=True)
axes[0].set_xlabel('P(GSR)')
axes[0].set_ylabel('Density')
axes[0].set_title('P(GSR) Distribution: Non-Shooter vs Shooter')
axes[0].legend()
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

axes[1].hist(nonshooter_probs, bins=100, alpha=0.7, edgecolor='black')
axes[1].set_xlabel('P(GSR)')
axes[1].set_ylabel('Count')
axes[1].set_title('Non-Shooter P(GSR) Distribution')
axes[1].axvline(x=thresholds['Default Threshold'], linestyle='--', alpha=0.7, label='Default threshold')
axes[1].legend()
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

axes[2].hist(shooter_probs, bins=100, alpha=0.7, edgecolor='black')
axes[2].set_xlabel('P(GSR)')
axes[2].set_ylabel('Count')
axes[2].set_title('Shooter P(GSR) Distribution')
axes[2].axvline(x=thresholds['Default Threshold'], linestyle='--', alpha=0.7, label='Default threshold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 13. Summary Statistics


In [ ]:
print('Summary stats')
print(f'
Non-shooter:')
print(f'N: {len(nonshooter_probs):,}')
print(f'Mean P(GSR): {nonshooter_probs.mean():.6f}')
print(f'Median P(GSR): {np.median(nonshooter_probs):.6f}')
print(f'P(GSR) < 0.1: {(nonshooter_probs < 0.1).mean():.2%}')
print(f'P(GSR) > 0.5: {(nonshooter_probs > 0.5).mean():.2%}')

print(f'
Shooter:')
print(f'N: {len(shooter_probs):,}')
print(f'Mean P(GSR): {shooter_probs.mean():.6f}')
print(f'Median P(GSR): {np.median(shooter_probs):.6f}')
print(f'P(GSR) < 0.1: {(shooter_probs < 0.1).mean():.2%}')
print(f'P(GSR) > 0.5: {(shooter_probs > 0.5).mean():.2%}')

nonshooter_gsr_rate = (nonshooter_probs >= thresholds['Default Threshold']).mean()
shooter_gsr_rate = (shooter_probs >= thresholds['Default Threshold']).mean()

print(f'
Contrast at default threshold = {thresholds["Default Threshold"]:.2f}:')
print(f'Non-shooter GSR rate: {nonshooter_gsr_rate:.4%}')
print(f'Shooter GSR rate: {shooter_gsr_rate:.2%}')
print(f'Ratio: {shooter_gsr_rate / max(nonshooter_gsr_rate, 1e-10):.1f}x')


## 14. Interpretation Markdown Template

The logistic regression model performed strongly on the original NFI held-out test set, but the NIST validation provides a more demanding external check. The non-shooter NIST particles act as a specificity test because any particles classified as GSR are false positives. The shooter NIST particles provide a sensitivity contrast because a model that generalizes well should assign higher GSR probabilities to shooter particles than to brake dust or fireworks particles.

At the default threshold of 0.50, the NIST false-positive rate should be reviewed first. If the non-shooter false-positive rate is high, this suggests that the model is learning element-pattern similarity that does not transfer cleanly across datasets. If the shooter GSR rate is not meaningfully higher than the non-shooter GSR rate, this would indicate poor cross-dataset separation.

Because `pb`, `sb`, and `ba` were excluded from the logistic regression model, this validation is especially useful for evaluating whether the remaining numeric feature space can generalize beyond the NFI dataset. The source-level and class-level breakdowns should be used to identify whether false positives are concentrated in specific NIST sources, such as brake dust or fireworks, rather than spread evenly across all non-shooter particles.
